# Практикум 5. Статическая векторизация: BoW, TF-IDF, Word2Vec, FastText

В лаборатории 5 вы видели матрицу «документ — слово», обучение skip-gram на игрушечном корпусе и символьные n-граммы FastText. Здесь повторим всё это в коде:

1. построим мешок слов (BoW) и TF-IDF вручную, сверим с `scikit-learn` и сравним документы по косинусному сходству;
2. напишем skip-gram с отрицательными примерами (SGNS) на `numpy` и обучим его на корпусе из 38 предложений;
3. обучим `Word2Vec` из библиотеки `gensim` на том же корпусе, сравним соседей и разберём аналогии «король − мужчина + женщина»;
4. разберём FastText: символьные n-граммы и векторы для форм, которых не было в корпусе («столами»).

*Практикум к лаборатории 5 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

In [ ]:
# gensim нет в Colab по умолчанию: ставим (около 30 секунд)
!pip install -q gensim

## 1. Мешок слов (BoW) и TF-IDF

Документ превращается в вектор длины $|V|$ (размер словаря). В **мешке слов** координата — сколько раз слово встретилось в документе, порядок слов теряется. Слова, которые есть во многих документах («на», «кухне»), мало говорят о содержании, поэтому **TF-IDF** умножает частоту на обратную документную частоту:

$$\text{tfidf}(w,d)=\text{tf}(w,d)\cdot\text{idf}(w),\qquad \text{idf}(w)=\ln\frac{1+N}{1+\text{df}(w)}+1,$$

где $N$ — число документов, $\text{df}(w)$ — в скольких документах есть слово $w$. Это вариант scikit-learn (`smooth_idf=True`); после умножения каждый вектор-строку делят на её длину (L2-норму).

In [ ]:
import re
import numpy as np
import pandas as pd

docs = [
    "Кошка спит на диване, кошка любит диван.",
    "Собака спит во дворе.",
    "Кошка и собака играют во дворе.",
    "Мама варит суп на кухне.",
    "Папа варит кофе на кухне.",
    "На кухне пахнет супом и кофе.",
]

def tokenize(text):
    # слова: последовательности букв и цифр, в нижнем регистре
    return re.findall(r"\w+", text.lower())

tok_docs = [tokenize(d) for d in docs]
vocab = sorted({w for d in tok_docs for w in d})
w2i = {w: i for i, w in enumerate(vocab)}
N, V = len(docs), len(vocab)
names = ["d" + str(i + 1) for i in range(N)]

# BoW: матрица N x V, в клетке — сколько раз слово встретилось в документе
counts = np.zeros((N, V))
for i, d in enumerate(tok_docs):
    for w in d:
        counts[i, w2i[w]] += 1

# IDF как в scikit-learn (smooth_idf=True)
df = (counts > 0).sum(axis=0)
idf = np.log((1 + N) / (1 + df)) + 1
tfidf = counts * idf
tfidf = tfidf / np.linalg.norm(tfidf, axis=1, keepdims=True)   # L2-нормировка строк

for name, d in zip(names, docs):
    print(name, d)
table = pd.DataFrame(counts.T.astype(int), index=vocab, columns=names)
table["df"] = df
table["idf"] = idf.round(2)
print()
print(table.to_string())

print("TF-IDF (столбцы — документы):")
print(pd.DataFrame(tfidf, index=names, columns=vocab).round(2).T.to_string())
print()
for i in range(N):
    top = np.argsort(-tfidf[i], kind="stable")[:3]
    print(names[i], "самые весомые слова:", ", ".join(f"{vocab[j]} ({tfidf[i, j]:.2f})" for j in top))

Те же матрицы даёт scikit-learn. Чтобы словари совпали, передадим векторизаторам нашу функцию `tokenize` (по умолчанию sklearn выбрасывает однобуквенные слова вроде «и»).


**Косинусное сходство** документов: $\cos(a,b)=\dfrac{a\cdot b}{\|a\|\,\|b\|}$. Сравним матрицы сходства для BoW и TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cv = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
X_bow = cv.fit_transform(docs).toarray()
tv = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
X_tfidf = tv.fit_transform(docs).toarray()

print("словарь совпадает: ", list(cv.get_feature_names_out()) == vocab)
print("BoW совпадает:     ", np.array_equal(X_bow, counts))
print("IDF совпадает:     ", np.allclose(tv.idf_, idf))
print("TF-IDF совпадает:  ", np.allclose(X_tfidf, tfidf))

import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

def cosine_matrix(X):
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    return Xn @ Xn.T

S_bow, S_tfidf = cosine_matrix(counts), cosine_matrix(tfidf)
print("совпадает с sklearn:", np.allclose(S_tfidf, cosine_similarity(X_tfidf)))
print("\nBoW:")
print(pd.DataFrame(S_bow, index=names, columns=names).round(2).to_string())
print("\nTF-IDF:")
print(pd.DataFrame(S_tfidf, index=names, columns=names).round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, S, title in zip(axes, [S_bow, S_tfidf], ["BoW", "TF-IDF"]):
    ax.imshow(S, cmap="Blues", vmin=0, vmax=1)
    for i in range(N):
        for j in range(N):
            ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if S[i, j] > 0.6 else "black")
    ax.set_xticks(range(N), names)
    ax.set_yticks(range(N), names)
    ax.set_title("Косинусное сходство документов, " + title)
plt.tight_layout()
plt.show()

Обратите внимание на пару d4–d5 («Мама варит суп на кухне» и «Папа варит кофе на кухне»): в BoW её сходство 0,60 — у документов три общих слова «варит», «на», «кухне». В TF-IDF эти слова встречаются в нескольких документах и получают меньший вес, а редкие «мама», «папа», «суп», «кофе» — больший, поэтому сходство падает до 0,45. Сходство d1 с d4 и d5 держится только на служебном «на»: в TF-IDF оно падает с 0,15 до 0,07–0,08. Пары без общих слов (d2–d4, d2–d5 и другие) остаются нулевыми в обеих моделях: ни одна из них не знает, что «кошка» и «собака» похожи по смыслу, и даже что «суп» (d4) и «супом» (d6) — формы одного слова. Эту задачу решают векторы слов.

## 2. Word2Vec skip-gram с отрицательными примерами: с нуля на numpy

Идея дистрибутивной семантики: слова, которые встречаются в похожих контекстах, близки по смыслу. **Skip-gram** берёт центральное слово $c$ и учится предсказывать слова $o$ в окне вокруг него. У каждого слова два вектора: $v_w$ (как центр) и $u_w$ (как контекст). Вместо softmax по всему словарю используем **отрицательные примеры** (negative sampling): для настоящей пары $(c, o)$ берём $K$ случайных слов $n_1,\dots,n_K$ и минимизируем

$$L = -\log\sigma(u_o^\top v_c) - \sum_{k=1}^{K}\log\sigma(-u_{n_k}^\top v_c),\qquad \sigma(x)=\frac{1}{1+e^{-x}}.$$

Случайные слова берутся с вероятностью $P(w)\propto \text{freq}(w)^{0.75}$: степень 0,75 немного поднимает редкие слова. Корпус ниже: 38 коротких предложений про семью, животных, еду и мебель.


Сначала разрежем корпус на пары (центр, контекст) скользящим окном, как в анимации лаборатории.

In [ ]:
from collections import Counter

corpus = """
мама читает книгу дома
папа читает газету дома
бабушка читает сказку внукам
брат пишет письмо дома
сестра пишет письмо бабушке
мама и папа работают утром
брат и сестра учатся утром
папа говорит с мамой по телефону
кошка спит во дворе
собака спит во дворе
собака бегает во дворе
кошка бегает по траве
корова ест траву на лугу
лошадь ест траву на лугу
коза пьёт воду на лугу
лошадь пьёт воду у реки
корова пьёт воду у реки
кошка пьёт тёплое молоко
мама варит вкусный суп
бабушка варит вкусный плов
папа покупает свежий хлеб
сестра покупает свежий сыр
брат ест горячий суп с хлебом
на обед мы едим плов и хлеб
на ужин мы едим суп и сыр
свежий хлеб лежит на столе
новый стол стоит в комнате
старый шкаф стоит в комнате
мягкий диван стоит у окна
деревянный стул стоит у стола
папа чинит старый стул
мама купила новый диван
за столом сидит вся семья
кошка спит под столом
брат подвинул стол к окну
книги лежат в шкафу
новые столы стоят в классе
собака подошла к столу
"""

sents = [tokenize(line) for line in corpus.strip().splitlines()]
freq = Counter(w for s in sents for w in s)
words = sorted(freq)                      # словарь корпуса
idx = {w: i for i, w in enumerate(words)}
print(len(sents), "предложений,", sum(freq.values()), "словоупотреблений,", len(words), "разных слов")
print("самые частые:", freq.most_common(8))

WINDOW = 2

def context_positions(sent, i, window):
    return [j for j in range(max(0, i - window), min(len(sent), i + window + 1)) if j != i]

# окно скользит по первому предложению
s0 = sents[0]
for i, c in enumerate(s0):
    print(f"{c:>12} → {[s0[j] for j in context_positions(s0, i, WINDOW)]}")

pairs = np.array([(idx[s[i]], idx[s[j]]) for s in sents for i in range(len(s))
                  for j in context_positions(s, i, WINDOW)])
print("\nвсего пар (центр, контекст):", len(pairs))

# распределение для отрицательных примеров: freq^0.75
f = np.array([freq[w] for w in words], dtype=float)
noise = f ** 0.75 / (f ** 0.75).sum()
for w in [w for w, _ in freq.most_common(3)] + [words[int(np.argmin(f))]]:
    print(f"   {w:<10} частота {f[idx[w]] / f.sum():.3f}   P_noise {noise[idx[w]]:.3f}")

Теперь обучение. Для каждой пары считаем $\sigma(u^\top v)$ для настоящего контекста и $K=5$ отрицательных слов, градиент по оценке равен $\sigma(u^\top v) - \text{label}$ (метка 1 у настоящей пары, 0 у случайных). Шаг SGD притягивает $v_c$ к $u_o$ и отталкивает от $u_{n_k}$. 100 эпох по 480 парам занимают около секунды.


Что выучила модель? $\sigma(u_o^\top v_c)$ — это её оценка вероятности того, что пара «настоящая». Сравним пары из корпуса и пары, которых в нём нет, а затем найдём ближайших соседей по косинусу векторов $v$.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def train_sgns(pairs, V, dim=10, neg=5, lr=0.05, epochs=100, seed=0):
    rng = np.random.default_rng(seed)
    W_in = (rng.random((V, dim)) - 0.5) / dim    # v: векторы центральных слов (их и берём как эмбеддинги)
    W_out = np.zeros((V, dim))                   # u: векторы контекстных слов
    labels = np.zeros(neg + 1)
    labels[0] = 1                                # 1 настоящая пара + neg случайных
    history = []
    for ep in range(epochs):
        negs = rng.choice(V, size=(len(pairs), neg), p=noise)
        total = 0.0
        for n in rng.permutation(len(pairs)):
            c, o = pairs[n]
            ids = np.concatenate(([o], negs[n]))
            v, u = W_in[c], W_out[ids]
            p = sigmoid(u @ v)
            g = p - labels                       # градиент потерь по оценкам u·v
            total += -np.log(p[0] + 1e-12) - np.log(1 - p[1:] + 1e-12).sum()
            W_in[c] -= lr * g @ u
            np.add.at(W_out, ids, -lr * np.outer(g, v))
        history.append(total / len(pairs))
        if ep % 20 == 0 or ep == epochs - 1:
            print(f"эпоха {ep:3d}   средние потери {history[-1]:.3f}")
    return W_in, W_out, history

W_in, W_out, history = train_sgns(pairs, len(words))

In [ ]:
def pair_prob(a, b):
    return sigmoid(W_out[idx[b]] @ W_in[idx[a]])

print("пары из корпуса:")
for a, b in [("кошка", "бегает"), ("суп", "вкусный"), ("корова", "траву"), ("стул", "старый")]:
    print(f"   σ(u_{b} · v_{a}) = {pair_prob(a, b):.2f}")
print("пар нет в корпусе:")
for a, b in [("кошка", "газету"), ("суп", "шкаф"), ("корова", "письмо"), ("стул", "реки")]:
    print(f"   σ(u_{b} · v_{a}) = {pair_prob(a, b):.2f}")

E = W_in / np.linalg.norm(W_in, axis=1, keepdims=True)

def neighbours(E, word, k=5):
    sims = E @ E[idx[word]]
    order = [j for j in np.argsort(-sims) if j != idx[word]][:k]
    return [(words[j], round(float(sims[j]), 2)) for j in order]

print()
for q in ["кошка", "суп", "хлеб", "мама", "стул"]:
    print(f"{q:>8}:", neighbours(E, q))

Соседи на таком маленьком корпусе шумные, но закономерность видна: животные оказываются рядом с местами и действиями животных («во дворе», «траве», «молоко»), еда — с едой («суп» — «плов», «хлеб» — «сыр»), мебель — с мебелью («стул» — «шкаф»). Нарисуем векторы на плоскости методом главных компонент (PCA).

In [ ]:
groups = {
    "семья": ["мама", "папа", "бабушка", "брат", "сестра"],
    "животные": ["кошка", "собака", "корова", "лошадь", "коза"],
    "еда": ["суп", "плов", "хлеб", "сыр", "молоко"],
    "мебель": ["стол", "стул", "шкаф", "диван"],
}

# PCA через SVD: проекция на две главные компоненты
Xc = W_in - W_in.mean(axis=0)
U_, S_, Vt = np.linalg.svd(Xc, full_matrices=False)
P2 = Xc @ Vt[:2].T
print("доля дисперсии в двух компонентах:", round(float((S_[:2] ** 2).sum() / (S_ ** 2).sum()), 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(history, color="#2B55C9")
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("средние потери на пару")
axes[0].set_title("Обучение SGNS")
colors = ["#2B55C9", "#D9532C", "#17907E", "#8E44AD"]
for (g, ws), col in zip(groups.items(), colors):
    pts = np.array([P2[idx[w]] for w in ws])
    axes[1].scatter(pts[:, 0], pts[:, 1], color=col, label=g)
    for w, (x, y) in zip(ws, pts):
        axes[1].annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[1].legend(fontsize=8)
axes[1].set_title("Векторы слов, PCA")
plt.tight_layout()
plt.show()

## 3. Word2Vec в gensim

Библиотека `gensim` реализует тот же алгоритм (`sg=1` — skip-gram, `negative=5` — пять отрицательных примеров) на C, со снижением шага обучения и прореживанием частых слов. На крошечном корпусе прореживание выключим (`sample=0`), а эпох возьмём больше. `workers=1` и `seed` делают результат воспроизводимым.

In [ ]:
import time
from gensim.models import Word2Vec

t0 = time.time()
w2v = Word2Vec(sents, vector_size=20, window=2, min_count=1, sg=1, negative=5,
               sample=0, epochs=300, seed=1, workers=1)
print(f"обучение заняло {time.time() - t0:.1f} s, словарь {len(w2v.wv)}, размерность {w2v.wv.vector_size}")

rows = []
for q in ["кошка", "суп", "хлеб", "мама", "стул"]:
    ours = [w for w, _ in neighbours(E, q)]
    lib = [w for w, _ in w2v.wv.most_similar(q, topn=5)]
    rows.append((q, ", ".join(ours), ", ".join(lib), len(set(ours) & set(lib))))
print(pd.DataFrame(rows, columns=["слово", "наш SGNS", "gensim", "общих"]).to_string(index=False))

**Аналогии.** Знаменитое «король − мужчина + женщина ≈ королева» работает на векторах, обученных на миллиардах слов: разность «женщина − мужчина» там примерно одинакова для многих пар. На нашем корпусе из 177 словоупотреблений аналогий не получится, поэтому возьмём маленький **набор векторов, заданный вручную** (как в лаборатории), с тремя осями: «царственность», «пол», «возраст». Метод `most_similar(positive=..., negative=...)` из gensim ищет слово, ближайшее к сумме векторов.


На графике стрелки «мужчина → женщина», «король → королева», «отец → мать» параллельны: это и есть «направление пола». Метод `most_similar` исключает из ответа сами входные слова.

In [ ]:
from gensim.models import KeyedVectors

an_words = ["король", "королева", "мужчина", "женщина", "отец", "мать", "сын", "дочь"]
#            царств. пол  возраст
an_vectors = [[0.9,  0.8, 0.7],   # король
              [0.9, -0.8, 0.7],   # королева
              [0.1,  0.8, 0.5],   # мужчина
              [0.1, -0.8, 0.5],   # женщина
              [0.2,  0.7, 0.9],   # отец
              [0.2, -0.7, 0.9],   # мать
              [0.1,  0.7, 0.1],   # сын
              [0.1, -0.7, 0.1]]   # дочь
queries = [(["король", "женщина"], ["мужчина"]),
           (["отец", "женщина"], ["мужчина"]),
           (["сын", "женщина"], ["мужчина"])]
ARROWS = [("мужчина", "женщина"), ("король", "королева"), ("отец", "мать")]

kv = KeyedVectors(vector_size=3)
kv.add_vectors(an_words, np.array(an_vectors, dtype=np.float32))

for pos, neg in queries:
    best = kv.most_similar(positive=pos, negative=neg, topn=2)
    expr = f"{pos[0]} − {neg[0]} + {pos[1]}"
    print(f"{expr:<28} ≈ {best[0][0]:<10} (cos {best[0][1]:.2f});  второе место: {best[1][0]} ({best[1][1]:.2f})")

# то же вручную: сумма векторов и косинус ко всем словам
A = np.array(an_vectors)
An = A / np.linalg.norm(A, axis=1, keepdims=True)
pos, neg = queries[0]
target = An[an_words.index(pos[0])] - An[an_words.index(neg[0])] + An[an_words.index(pos[1])]
sims = An @ target / np.linalg.norm(target)
cand = [(an_words[j], round(float(sims[j]), 2)) for j in np.argsort(-sims) if an_words[j] not in pos + neg]
print("вручную:", cand[:2])

fig, ax = plt.subplots(figsize=(5, 4))
# рисуем оси «пол» и «возраст»; размер точки — «царственность»
ax.scatter(A[:, 1], A[:, 2], s=40 + 200 * A[:, 0], color="#2B55C9")
for w, (r, g, a) in zip(an_words, A):
    ax.annotate(w, (g, a), fontsize=9, xytext=(0, 9), textcoords="offset points", ha="center")
for a, b in ARROWS:
    pa, pb = A[an_words.index(a)], A[an_words.index(b)]
    ax.annotate("", xy=(pb[1], pb[2]), xytext=(pa[1], pa[2]), arrowprops=dict(arrowstyle="->", color="#D9532C"))
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(0, 1.1)
ax.set_xlabel("ось «пол»")
ax.set_ylabel("ось «возраст»")
ax.set_title("Ручные векторы: «пол» — одно направление")
plt.tight_layout()
plt.show()

## 4. FastText: символьные n-граммы и незнакомые слова

У Word2Vec нет вектора для слова, которого не было в корпусе: «столами» для него просто неизвестная строка. **FastText** представляет слово набором символьных n-грамм: к слову добавляют границы `<` и `>`, и при $n=3\ldots5$ «стол» даёт `<ст`, `сто`, `тол`, `ол>`, `<сто`, `стол`, `тол>`, `<стол`, `стол>`. Вектор слова собирается из векторов его n-грамм (в gensim — среднее; для слова из корпуса добавляется ещё его собственный вектор). Для незнакомой формы вектор собирается из знакомых кусочков. Для языков с богатой морфологией (русский, а тем более узбекский) это особенно важно.

In [ ]:
from gensim.models.fasttext import compute_ngrams

def char_ngrams(word, min_n=3, max_n=5):
    w = "<" + word + ">"
    return [w[i:i + n] for n in range(min_n, max_n + 1) for i in range(len(w) - n + 1)]

word = "столами"
grams = char_ngrams(word)
print(word, "→", len(grams), "n-грамм:")
print("  ", grams)
print("совпадает с gensim:", sorted(grams) == sorted(compute_ngrams(word, 3, 5)))

# какие слова корпуса делят с ним больше всего n-грамм (коэффициент Жаккара)
g = set(grams)
share = sorted(((len(g & set(char_ngrams(w))) / len(g | set(char_ngrams(w))), w) for w in words), reverse=True)
print("ближайшие по n-граммам:", [(w, round(j, 2)) for j, w in share[:5]])

Обучим FastText в gensim на том же корпусе и попросим векторы для форм, которых в корпусе нет. Word2Vec на этих словах падает с `KeyError`.


Проверим, что gensim действительно складывает n-граммы: соберём вектор незнакомого слова сами как среднее векторов его n-грамм из `ft.wv.vectors_ngrams` (номер строки — хэш n-граммы, `ft_ngram_hashes`).

In [ ]:
from gensim.models import FastText

ft = FastText(sents, vector_size=20, window=2, min_count=1, sg=1, negative=5, sample=0,
              epochs=300, seed=1, workers=1, min_n=3, max_n=5, bucket=20000)

for w in ["столами", "кошками", "сабака"]:
    print(w, "| в корпусе:", w in ft.wv.key_to_index)
    try:
        w2v.wv[w]
    except KeyError:
        print("   Word2Vec: KeyError — вектора нет")
    print("   FastText:", [(x, round(s, 2)) for x, s in ft.wv.most_similar(w, topn=4)])

from gensim.models.fasttext import ft_ngram_hashes

w = "столами"
hashes = ft_ngram_hashes(w, ft.wv.min_n, ft.wv.max_n, ft.wv.bucket)
manual = ft.wv.vectors_ngrams[hashes].mean(axis=0)
print(len(hashes), "строк из таблицы n-грамм, первые номера:", hashes[:5])
print("вектор вручную совпадает с ft.wv[w]:", np.allclose(manual, ft.wv[w], atol=1e-6))

У символьных n-грамм есть и обратная сторона: «столица» делит с «стол» начало `<сто`, `стол`, `<стол`, хотя по смыслу это разные слова. Опечатка «сабака», наоборот, полезно оказывается рядом с «собака». На графике сравним сходство незнакомых форм с их «родными» словами и это ложное сходство.

In [ ]:
pairs_plot = [("столами", "стол"), ("столах", "стол"), ("кошками", "кошка"),
              ("собаки", "собака"), ("сабака", "собака"), ("столица", "стол")]
FALSE_FRIEND = "столица"

sims = [float(ft.wv.similarity(a, b)) for a, b in pairs_plot]
labels = [f"{a}\n~ {b}" for a, b in pairs_plot]
for (a, b), s in zip(pairs_plot, sims):
    print(f"{a:>17} ~ {b:<8} cos = {s:.2f}   (в корпусе: {a in ft.wv.key_to_index})")

fig, ax = plt.subplots(figsize=(9, 3.6))
cols = ["#D9532C" if a == FALSE_FRIEND else "#17907E" for a, _ in pairs_plot]
ax.bar(labels, sims, color=cols)
ax.set_ylim(0, 1)
ax.set_ylabel("косинус FastText")
ax.set_title("Незнакомые формы и их слова (красным — ложное сходство)")
ax.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()

## 5. Задания

1. **Сублинейный TF.** Посчитайте TF-IDF вручную с $\text{tf}=1+\ln(\text{count})$ (для ненулевых счётчиков) и сверьте с `TfidfVectorizer(sublinear_tf=True, tokenizer=tokenize, lowercase=False, token_pattern=None)`. Для какого документа изменения заметнее всего и почему?
2. **Ширина окна.** Обучите наш SGNS (раздел 2) с `WINDOW = 1` и `WINDOW = 4` и сравните пять ближайших соседей слов «суп» и «стол». Какое окно даёт более «тематических», а какое более «грамматических» соседей?
3. **Новая тема.** Добавьте в `corpus` 6–8 предложений про транспорт (автобус, поезд, машина, едет, вокзал…) и обучите `Word2Vec` из раздела 3 заново. Попадают ли новые слова в соседи друг к другу? Как меняется ответ, если поставить `epochs=50` вместо 300?
4. **FastText и морфология.** Придумайте пять форм слов корпуса, которых в нём нет («собаками», «кошкам», «столов», «маме», «хлебу»), и для каждой выведите ближайшее слово по FastText. Найдите пример, где FastText ошибается из-за общих букв.
5. **Длина n-грамм.** Обучите FastText с `min_n=2, max_n=3` и с `min_n=4, max_n=6`. Как меняется сходство «столами» — «стол» и «столица» — «стол»? Объясните.

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь